# Analysis of perturbing and Doppler-Broadening cross sections

In [ ]:
import sandy
import pandas as pd

In [ ]:
import matplotlib.pyplot as plt

# Create perturbed PENDF XS

## Retrieve ENDF-6 file

In [ ]:
tape = sandy.get_endf6_file("endfb_80", "xs", 942390)

In [ ]:
pendf = tape.get_pendf(err=0.001)
xs = sandy.Xs.from_endf6(pendf)

## Extract cross sections and perturb after RECONR

In [ ]:
pert_coeff = 1
ipert = 25

egrid = sandy.energy_grids.SCALE238
estart = egrid[ipert]
estop = egrid[ipert+1]

pert = sandy.Pert([1, 1 + pert_coeff], index=[estart, estop])

print(
    f"""
Perturb fission xs in range ({estart}, {estop}) eV by a factor {pert_coeff*100} %.
Perturbation object is:
{pert}
"""
 )

In [ ]:
mat = tape.mat[0]
mt = 18

xspert = xs.custom_perturbation(mat, mt, pert)

## Analyize how the perturbation was implemented

In [ ]:
def apply_mask(xs, estart, estop):
    mask = (xs.data.index >= estart) & (xs.data.index <= estop)
    return xs.data.loc[mask]

In [ ]:
fig, ax = plt.subplots()

ax = xspert.data[mat, mt].plot(marker="s", ax=ax, ls="--", lw=.5, label=f"RECONR + PERT")

ax = xs.data[mat, mt].plot(ls="--", lw=.5, c="k", ax=ax, label=f"RECONR")

ax.set(xlim=(0.15, 0.25), ylim=(300, 2000), xlabel='energy / $eV$', ylabel="fission xs / $b$")
ax.legend(loc=4)

fig.tight_layout()

## Write PENDF files

In [ ]:
pendf.to_file("pendffile")

pendfpert = xspert.to_endf6(pendf).update_intro()
pendfpert.to_file("pert_pendffile")

## Doppler-Broaden (BROADR) original and perturbed files

In [ ]:
broadr = tape.get_pendf(
    pendftape="pendffile",
    err=0.001,
    temperature=900,
    heatr=False, purr=False, groupr=False, thermr=False, gaspr=False,
)

broadrpert = tape.get_pendf(
    pendftape="pert_pendffile",
    err=0.001,
    temperature=900,
    heatr=False, purr=False, groupr=False, thermr=False, gaspr=False,
)

## Extract and plot cross sections

In [ ]:
xsbroadr = sandy.Xs.from_endf6(broadr)
xsbroadrpert = sandy.Xs.from_endf6(broadrpert)

In [ ]:
fig, ax = plt.subplots()

ax = xsbroadr.data[mat, mt].plot(marker="s", ax=ax, ls="--", lw=.5, label=f"RECONR + BROADR")

ax = xsbroadrpert.data[mat, mt].plot(marker="s", ax=ax, ls="--", lw=.5, label=f"RECONR + PERT + BROADR")

ax.set(xlim=(0.15, 0.25), ylim=(300, 2000), xlabel='energy / $eV$', ylabel="fission xs / $b$")
ax.legend(loc=4)

fig.tight_layout()

## Quick fix: apply perturbation after BROADR

In [ ]:
xsbroadrpert_fix = xsbroadr.custom_perturbation(mat, mt, pert)

In [ ]:
fig, ax = plt.subplots()

ax = xsbroadr.data[mat, mt].plot(marker="s", ax=ax, ls="--", lw=.5, label=f"RECONR + BROADR")

ax = xsbroadrpert.data[mat, mt].plot(marker="s", ax=ax, ls="--", lw=.5, label=f"RECONR + PERT + BROADR")

ax = xsbroadrpert_fix.data[mat, mt].plot(marker="s", ax=ax, ls="--", lw=.5, label=f"RECONR + BROADR + PERT")

ax.set(xlim=(0.15, 0.25), ylim=(300, 2000), xlabel='energy / $eV$', ylabel="fission xs / $b$")
ax.legend(loc=4)

fig.tight_layout()

## Get an ACE file with the quick fix

In [ ]:
ace = tape.get_ace(
    pendf=broadrpert_fix,
    err=0.001,
    temperature=900,
    broadr=False,
    heatr=False, purr=False, groupr=False, thermr=False, gaspr=False,
    verbose=True,
)

## Conclusions

When applying perturbations to a PENDF file **after RECONR**, but **before BROADR** (default SANDY implementation), the perturbation is effectively treatened as a resonance of the cross section and it is smoothened out by BROADR. 

An arguably more correct way is to apply the perturbation after doppler-broadening the cross sections (as explained above).